# Superstars Analytics Master

In [30]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [1]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [32]:
# Time
start = dt.datetime(2019,5,12)
end = dt.datetime(2019,5,13)
print(start,end,end-start)

2019-05-12 00:00:00 2019-05-13 00:00:00 1 day, 0:00:00


In [34]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
df = df[df["sign_up_details_app_platform"] == "UNITY_Android"]
#df = df[df["sign_up_details_device_id"].isin(devices)]
users = df[["_id","created_at","sign_up_details_device_id","login_details_last_request_at"]]
users.columns = ["user_id","createtime","device_id","last_request"]
len(users)

7423

In [37]:
users["last_request"].isnull().value_counts()

False    7396
True       27
Name: last_request, dtype: int64

In [39]:
team_cursor = cursor.superstars.teams
aw = []
for documents in team_cursor.find({'created_at': {'$lt': end, '$gte': start}},{"user":1, "created_at":1}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
teams = df[df["user"].isin(users["user_id"])]
teams = teams[["_id","user","created_at"]]
teams.columns = ["team_id", "user_id", "team_created_at"]
teams.head()

,team_id,user_id,team_created_at
0,5cd7620ed4288f6f1e35f8e6,5cd7620ed4288f6f1e35f8de,2019-05-12 00:00:14.610
1,5cd762a65f50b03a9dc9742a,5cd762a65f50b03a9dc97422,2019-05-12 00:02:46.429
2,5cd762b2cfd59d105c9b4ea2,5cd762b2cfd59d105c9b4e9a,2019-05-12 00:02:58.579
3,5cd762c062ebdd106378e6a3,5cd762c062ebdd106378e69b,2019-05-12 00:03:12.893
4,5cd762c74e76e0104100ca84,5cd762c74e76e0104100ca7c,2019-05-12 00:03:19.810


In [40]:
con_cursor = cursor.superstars.players
aw = []
for documents in con_cursor.find({'created_at': {'$lt': end, '$gte': start}, 'type' : 'STAR'},
                                 {"team":1, "created_at":1}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
players = df[df["team"].isin(teams["team_id"])]
players.columns = ["player_id","player_created_at","team_id"]
players.head()

,player_id,player_created_at,team_id
0,5cd7620ed4288f6f1e35f8ea,2019-05-12 00:00:14.620,5cd7620ed4288f6f1e35f8e6
1,5cd7620ed4288f6f1e35f8ee,2019-05-12 00:00:14.825,5cd7620ed4288f6f1e35f8e6
3,5cd762a65f50b03a9dc9742e,2019-05-12 00:02:46.434,5cd762a65f50b03a9dc9742a
4,5cd762a65f50b03a9dc97432,2019-05-12 00:02:46.639,5cd762a65f50b03a9dc9742a
5,5cd762b2cfd59d105c9b4ea6,2019-05-12 00:02:58.585,5cd762b2cfd59d105c9b4ea2
